# 07. SQLChatMessageHistory + RunnableWithMessageHistory → `SqliteSaver` (영속 checkpointer)

| legacy | LangGraph |
|---|---|
| `RunnableWithMessageHistory(chain, get_session_history, ...)` | `builder.compile(checkpointer=...)` |
| `store = {}` + `ChatMessageHistory()` (메모리) | `InMemorySaver()` |
| `SQLChatMessageHistory(connection="sqlite:///sqlite.db")` | `SqliteSaver(sqlite3.connect("....db"))` (`pip install langgraph-checkpoint-sqlite`) |
| `config={"configurable": {"session_id": ...}}` | `config={"configurable": {"thread_id": ...}}` |
| `ConfigurableFieldSpec(user_id / conversation_id)` + `history_factory_config` | `thread_id` 를 조합해서 사용 (예: `f"{user_id}:{conversation_id}"`) / 사용자 정보는 `context=` 로 전달 |
| `input_messages_key`, `history_messages_key` | 불필요 (`state["messages"]` 하나로 통일) |

> `RunnableWithMessageHistory` 경고 메시지: **"Use LangGraph's built-in persistence instead."**
>
> 참고로 `SQLChatMessageHistory` 클래스 자체는 `langchain_community` 에 **그대로 남아 있습니다.** (마지막 섹션에서 기존 데이터를 이관할 때 사용)

In [1]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

In [2]:
import os
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver

DB_PATH = "langgraph_checkpoints.db"
if os.path.exists(DB_PATH):   # 실습을 처음부터 다시 하기 위해 초기화
    os.remove(DB_PATH)

conn = sqlite3.connect(DB_PATH, check_same_thread=False)
checkpointer = SqliteSaver(conn)   # 테이블은 처음 사용할 때 자동 생성

## 1. 프롬프트/체인 (legacy 와 동일) + 그래프

In [3]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage
from langgraph.graph import StateGraph, MessagesState, START

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant."),
        MessagesPlaceholder(variable_name="chat_history"),
        ("human", "{question}"),
    ]
)
chain = prompt | llm


def chatbot(state: MessagesState):
    *history, last = state["messages"]
    return {"messages": [chain.invoke({"chat_history": history, "question": last.content})]}


def build_graph(saver):
    return (
        StateGraph(MessagesState)
        .add_node("chatbot", chatbot)
        .add_edge(START, "chatbot")
        .compile(checkpointer=saver)
    )


graph = build_graph(checkpointer)

## 2. user_id + conversation_id 로 대화 구분
legacy 의 `ConfigurableFieldSpec` 2개 + `get_chat_history(user_id, conversation_id)` 팩토리가 `thread_id` 문자열 하나로 대체됩니다.

In [4]:
def make_config(user_id: str, conversation_id: str) -> dict:
    return {"configurable": {"thread_id": f"{user_id}:{conversation_id}"}}


def ask(question: str, config: dict) -> str:
    out = graph.invoke({"messages": [HumanMessage(question)]}, config)
    return out["messages"][-1].content


config = make_config("user1", "conversation1")
print(ask("안녕 반가워 내 이름은 테디야", config))
print(ask("내 이름이 뭐라고?", config))

안녕하세요, 테디! 반가워요. 어떻게 도와드릴까요?


당신의 이름은 테디라고 하셨어요. 맞나요?


In [5]:
# 다른 conversation → 다른 thread → 기억 없음
print(ask("내 이름이 뭐라고?", make_config("user1", "conversation2")))

죄송하지만, 당신의 이름을 알 수 있는 정보가 없습니다. 당신의 이름을 알려주시면 그에 맞춰 대화할 수 있습니다!


## 3. 영속성 확인: 연결을 닫고 새로 열어도 대화가 남아 있다 (프로그램 재시작 상황)

In [6]:
conn.close()

conn = sqlite3.connect(DB_PATH, check_same_thread=False)
graph = build_graph(SqliteSaver(conn))

for m in graph.get_state(make_config("user1", "conversation1")).values["messages"]:
    print(f"{type(m).__name__:12}|", m.content)

print("\n→", ask("우리가 처음에 무슨 얘기를 했지?", make_config("user1", "conversation1")))

HumanMessage| 안녕 반가워 내 이름은 테디야
AIMessage   | 안녕하세요, 테디! 반가워요. 어떻게 도와드릴까요?
HumanMessage| 내 이름이 뭐라고?
AIMessage   | 당신의 이름은 테디라고 하셨어요. 맞나요?



→ 처음에 당신은 "안녕 반가워 내 이름은 테디야"라고 말씀하셨고, 제가 당신을 반갑게 맞이했어요. 그 후에 당신이 제게 이름을 물어보셨죠. 더 이야기하고 싶은 주제가 있나요?


## 4. legacy 에는 없던 기능
* **체크포인트 이력**: 매 스텝마다 상태가 저장되므로 과거 시점을 조회하거나 그 시점에서 다시 실행(time-travel)할 수 있습니다.
* **thread 목록 조회**: 저장된 대화 목록을 checkpointer 에서 바로 조회합니다.

In [7]:
config = make_config("user1", "conversation1")
history = list(graph.get_state_history(config))
print("conversation1 의 체크포인트 수:", len(history))
for snap in history[:4]:
    print(f"  step={snap.metadata.get('step'):>2} | 메시지 수={len(snap.values.get('messages', []))} | next={snap.next}")

threads = {c.config["configurable"]["thread_id"] for c in graph.checkpointer.list(None)}
print("\n저장된 thread 목록:", threads)

conversation1 의 체크포인트 수: 9
  step= 7 | 메시지 수=6 | next=()
  step= 6 | 메시지 수=5 | next=('chatbot',)
  step= 5 | 메시지 수=4 | next=('__start__',)
  step= 4 | 메시지 수=4 | next=()

저장된 thread 목록: {'user1:conversation1', 'user1:conversation2'}


## 5. 기존 `SQLChatMessageHistory` 데이터를 LangGraph 로 이관하기
legacy 노트북이 만든 `../legacy/sqlite.db` 의 `user1` 테이블(conversation1)을 읽어서 새 thread 에 넣습니다. `SQLChatMessageHistory` 는 deprecated 가 아니므로 그대로 읽을 수 있습니다.

In [8]:
from langchain_community.chat_message_histories import SQLChatMessageHistory

legacy_history = SQLChatMessageHistory(
    table_name="user1",
    session_id="conversation1",
    connection="sqlite:///../legacy/sqlite.db",
)
old_messages = legacy_history.messages
print("legacy 메시지 수:", len(old_messages))

migrated = make_config("user1", "migrated-conversation1")
graph.update_state(migrated, {"messages": old_messages}, as_node="chatbot")
print(ask("내 이름이 뭐였지?", migrated))

C:\Users\user\AppData\Local\Temp\ipykernel_3212\4157497703.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.chat_message_histories import SQLChatMessageHistory


legacy 메시지 수: 4


당신의 이름은 테디입니다. 맞나요?


In [9]:
conn.close()